In [1]:
import os
import sys
import yaml
import pandas as pd

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

from Titanic.src.features.feature_selection import FeatureSelectionOrchestrator
from Titanic.src.utils.plots import Pearson_correlation, Bar_plot
from Titanic.src.features.feature_eng import PreprocessingOrchestrator

In [2]:
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
        
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)  

In [3]:
X_train = pd.read_parquet(
        os.path.join(
            config['init_path'],
            config['data']['processed'],
            "train_features.parquet")
    )
y_train = X_train[['Survived']]
    
preprocessor = PreprocessingOrchestrator(
        numerical_con=config_pipe['features']['num_con'], 
        numerical_dis=config_pipe['features']['num_dis'], 
        categorical_var=config_pipe['features']['cat_var'])
    
pipe = preprocessor.apply("preprocessing")
        
X_train_trans = pipe.fit_transform(X_train)

In [4]:
feature_selection = FeatureSelectionOrchestrator()

In [6]:
X_train_trans

,numerical_pipe_con__Age,numerical_pipe_dis__SibSp,numerical_pipe_dis__Parch,numerical_pipe_dis__IsAlone,categorical_pipe__Pclass,categorical_pipe__Sex,categorical_pipe__Ticket,categorical_pipe__Embarked,categorical_pipe__Title
0,22.0,1.0,0.0,0,3,male,A521171,S,Mr
1,38.0,1.0,0.0,0,1,female,PC17599,C,Mrs
2,26.0,0.0,0.0,1,3,female,STONO23101282,S,Miss
3,35.0,1.0,0.0,0,1,female,113803,S,Mrs
4,35.0,0.0,0.0,1,3,male,373450,S,Mr
...,...,...,...,...,...,...,...,...,...
886,27.0,0.0,0.0,1,2,male,211536,S,Rare
887,19.0,0.0,0.0,1,1,female,112053,S,Miss
888,28.0,1.0,2.0,0,3,female,WC6607,S,Miss
889,26.0,0.0,0.0,1,1,male,111369,C,Mr


In [5]:
QuiSquare = feature_selection.apply(
        "QuiSquare", 
        X_train_trans.filter(like='categorical'), 
        y_train)

Feature: categorical_pipe__Pclass
Feature: categorical_pipe__Sex
Feature: categorical_pipe__Ticket
Feature: categorical_pipe__Embarked
Feature: categorical_pipe__Title


In [6]:
QuiSquare

categorical_pipe__Title       3.957861e-61
categorical_pipe__Sex         1.197357e-58
categorical_pipe__Pclass      4.549252e-23
categorical_pipe__Embarked    1.618719e-06
categorical_pipe__Ticket      1.152730e-02
Name: QuiSquare, dtype: float64

In [12]:
Anova = feature_selection.apply(
        "Anova",
        X_train_trans.filter(like='numerical'),
        y_train)
    
mi = feature_selection.apply(
        "MutualInformationClassif", 
        X_train_trans.filter(like='numerical'), 
        y_train)
    
corr = feature_selection.apply(
        "PearsonCorrelation", 
        X_train_trans.filter(like='numerical'), 
        y_train)

c:\Users\gustavo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\gustavo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [9]:
Anova

numerical_pipe_dis__IsAlone    9.009490e-10
numerical_pipe_dis__Parch      1.479925e-02
numerical_pipe_con__Age        5.276069e-02
numerical_pipe_dis__SibSp      2.922439e-01
Name: Anova, dtype: float64

In [10]:
mi

numerical_pipe_dis__SibSp      0.035510
numerical_pipe_dis__IsAlone    0.032838
numerical_pipe_con__Age        0.011977
numerical_pipe_dis__Parch      0.003646
Name: mutual information, dtype: float64

In [13]:
corr

,numerical_pipe_con__Age,numerical_pipe_dis__SibSp,numerical_pipe_dis__Parch,numerical_pipe_dis__IsAlone
numerical_pipe_con__Age,1.000000,-0.233296,-0.172482,0.171647
numerical_pipe_dis__SibSp,-0.233296,1.000000,0.414838,-0.584471
numerical_pipe_dis__Parch,-0.172482,0.414838,1.000000,-0.583398
numerical_pipe_dis__IsAlone,0.171647,-0.584471,-0.583398,1.000000
